#### permutation_tests.ipynb
This notebook imports time-dependent continental flooding map data (provided as a binary state: 0=submerged/1=exposed) referenced to h3 grids, permutes them (randomly rotates them) and then uses logistic regression to perform TPW inversions on them (by looking for correlations with a $Y_{21}$ pattern on rotation). Many iterations of this process yields a population of results with which the statistical significance of the results of the same test on the original (unrotated) data can be established. Prior to permutation of the data, the observational records (per timestep) are clustered so as to group proximal observations together. 

The notebook also outputs a figure showing the distribution of the log likelihoods of the permuted models.

In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import methods as m

In [2]:
### user-specified inputs ###
eps = 2.0                # length scale over which to apply clustering (degrees)
permutations = 10     # number of permutations to execute per time-step
start = 0                # start time (in Ma) = youngest reconstruction step
stop = 320               # stop time (in Ma) = oldest reconstruction step
step = 10                # temporal step size (in Ma)

apply_tectonic_filter = False # if True: observations from newly formed passive margins and active foreland basins are removed
apply_glacial_filter = False  # if True: observations from Antarctica since 30 Ma are removed

datdir = f'data/permutations/eps_{eps}'  # directory to output permutation data
figdir = f'figs/permutations/eps_{eps}'  # directory to output permutation figs
if apply_tectonic_filter or apply_glacial_filter:
    datdir += '/filtered'               # alternate path for filtered data
os.makedirs(datdir, exist_ok=True)
os.makedirs(figdir, exist_ok=True)

In [3]:
# cycle over time-steps
for t in range(start, stop, step):
    print (f'processing interval {t}-{t+step} Ma')
    t0 = pd.read_csv(f'obs_grids/{t}_Ma.csv')               # read in observation grids
    t1 = pd.read_csv(f'obs_grids/{t+step}_Ma.csv')
    merged = pd.merge(t0, t1, on='name', suffixes=('_t0', '_t1'))   # merge dataframes
    merged = merged.dropna(subset=['state_t0', 'state_t1'])         # drop NaNs
    if apply_tectonic_filter:
        merged = merged[~(merged['in_margin_t0'] | merged['in_margin_t1'] | merged['in_basin_t0']  | merged['in_basin_t1'])]  # get rid of any 'True' values
    if apply_glacial_filter and t <= 30:
        merged = merged[merged['lat_t0'] >= -60]                    # remove all points above -60 S latitude since 30 Ma
    merged['diff'] =  merged['state_t1'] - merged['state_t0']       # find points where there is flux: -1 = regression / 1 = transgression

    fluxpts = merged[merged['diff'] != 0]    # isolate points where there is flux
    lats = fluxpts['lat_t0'].to_numpy()
    lons = fluxpts['lon_t0'].to_numpy()
    flux = fluxpts['diff'].to_numpy()
    y = (flux == 1).astype(int)              # convert -1/1 range to 0/1 for logistic regression

    clusters, n_clusters, noise = m.cluster(lats, lons, eps=eps, min_samples=3)       # cluster observation points according to eps value (effectively 'degrees')
    perm_results = m.permutation_test_parallelized(y, lats, lons, clusters, n=permutations, n_jobs=1)  # run n permutations of inversion analysis with randomly reassigned signs to each cluster
    
    df = pd.DataFrame(perm_results, columns=['llr'])    # keep the delta loglikelihood of each result
    df['n_clusters'] = n_clusters                       # log the number of clusters (a reflection of the validity of the analysis)
    df.to_csv(f'{datdir}/{t}_Ma.csv', index=False)

    fig = m.plot_permutation_results(t, perm_results, n_clusters, noise)    # plot the distribution of the best-fit results from each permutation test and the 95% percentile from the ensemble
    plt.savefig(f'{figdir}/{t}_Ma.png', bbox_inches='tight')
    plt.close()

processing interval 0-10 Ma
processing interval 10-20 Ma
processing interval 20-30 Ma
processing interval 30-40 Ma
processing interval 40-50 Ma
processing interval 50-60 Ma
processing interval 60-70 Ma
processing interval 70-80 Ma
processing interval 80-90 Ma
processing interval 90-100 Ma
processing interval 100-110 Ma
processing interval 110-120 Ma
processing interval 120-130 Ma
processing interval 130-140 Ma
processing interval 140-150 Ma
processing interval 150-160 Ma
processing interval 160-170 Ma
processing interval 170-180 Ma
processing interval 180-190 Ma
processing interval 190-200 Ma
processing interval 200-210 Ma
processing interval 210-220 Ma
processing interval 220-230 Ma
processing interval 230-240 Ma
processing interval 240-250 Ma
processing interval 250-260 Ma
processing interval 260-270 Ma
processing interval 270-280 Ma
processing interval 280-290 Ma
processing interval 290-300 Ma
processing interval 300-310 Ma
processing interval 310-320 Ma
